# Fantasy Basketball Rankings (Polars)

This notebook builds category-based fantasy basketball rankings using:
- Polars
- Volume-weighted FG% and FT%
- Robust z-score aggregation
- Turnovers excluded

### Import dependencies and set working directory

In [ ]:
import os
from pathlib import Path
import polars as pl

# Change to project root (update this path for your setup)
os.chdir(r"fantasy-basketball-rankings")


### Import and refine example stats

In [98]:
# Read example stats CSV into a Polars DataFrame
df = pl.read_csv('data/raw/example_stats.csv')

# Select and rename relevant columns
df = df.select(["Name","Pos","g","m","USG","p","3","r","a","s","b","to","fg%","ft%","fga","fta"])

df = df.rename({"g": "Games", "m": "Min", "p": "PTS", "3": "Threes", "r": "REB", "a": "AST", "s": "STL", "b": "BLK", "to": "TO", "fg%": "Field_Goal_Percentage", "ft%": "Free_Throw_Percentage", "fga": "Field_Goal_Attempts", "fta": "Free_Throw_Attempts"})

# Minimum games threshold
MIN_GAMES = 10

df = (
    df
    .filter(pl.col("Games") >= MIN_GAMES)
)

# View DataFrame
df

Name,Pos,Games,Min,USG,PTS,Threes,REB,AST,STL,BLK,TO,Field_Goal_Percentage,Free_Throw_Percentage,Field_Goal_Attempts,Free_Throw_Attempts
str,str,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64
"""Nikola Jokic""","""C""",27,940.9,29.961517,794,56,327,289,37,20,96,0.605485,0.841026,474,195
"""Shai Gilgeous-Alexander""","""PG""",27,896.4,33.652491,878,60,126,174,39,22,52,0.554307,0.8828125,534,256
"""Tyrese Maxey""","""PG""",25,995.366667,30.319171,792,95,113,178,42,22,65,0.467354,0.884393,582,173
"""Scottie Barnes""","""SG/SF/PF""",30,1016.566667,24.024475,572,40,240,152,41,50,79,0.497768,0.819048,448,105
"""Jalen Johnson""","""SF/PF""",27,948.966667,26.732867,642,44,284,221,40,10,93,0.524887,0.832298,442,161
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Daniel Gafford""","""C""",18,386.016667,15.062784,147,0,108,22,15,29,28,0.613636,0.795918,88,49
"""Tristan da Silva""","""SF/PF""",26,602.166667,17.369541,252,42,101,30,21,6,25,0.439252,0.814815,214,27
"""Robert Williams""","""C""",18,284.183333,12.46021,107,1,108,17,12,27,11,0.680556,0.615385,72,13


### League Averages (Weighted by Attempts)

In [99]:
# Compute league average FG% and FT%
league_fg = (
    df.select(
        (pl.col("Field_Goal_Percentage") * pl.col("Field_Goal_Attempts")).sum()
        / pl.col("Field_Goal_Attempts").sum()
    )
    .item()
)

league_ft = (
    df.select(
        (pl.col("Free_Throw_Percentage") * pl.col("Free_Throw_Attempts")).sum()
        / pl.col("Free_Throw_Attempts").sum()
    )
    .item()
)


### Volume-Weighted Efficiency Impact

In [100]:
df = df.with_columns([
    (
        (pl.col("Field_Goal_Percentage") - league_fg)
        * pl.col("Field_Goal_Attempts")
    ).alias("FG_Impact"),

    (
        (pl.col("Free_Throw_Percentage") - league_ft)
        * pl.col("Free_Throw_Attempts")
    ).alias("FT_Impact"),
])


### Per-Game Normalization

In [101]:
counting_stats = [
    "PTS", "Threes", "REB", "AST", "STL", "BLK"
]

df = df.with_columns([
    (pl.col(c) / pl.col("Games")).alias(f"{c}_pg")
    for c in counting_stats
])

df = df.with_columns([
    (pl.col("FG_Impact") / pl.col("Games")).alias("FG_Impact_pg"),
    (pl.col("FT_Impact") / pl.col("Games")).alias("FT_Impact_pg"),
])


### Winsorization (5th-95th percentile)

In [102]:
def winsorize(col: str, df: pl.DataFrame, lo=0.05, hi=0.95):
    lower = df.select(pl.col(col).quantile(lo)).item()
    upper = df.select(pl.col(col).quantile(hi)).item()

    return (
        pl.when(pl.col(col) < lower).then(lower)
        .when(pl.col(col) > upper).then(upper)
        .otherwise(pl.col(col))
    )

z_base_cols = [
    "PTS_pg", "Threes_pg", "REB_pg", "AST_pg",
    "STL_pg", "BLK_pg", "FG_Impact_pg", "FT_Impact_pg"
]

df = df.with_columns([
    winsorize(c, df).alias(c)
    for c in z_base_cols
])

### Compute Z-Scores Per Category

In [103]:
Z_COLS = {}

for metric in CATEGORY_WEIGHTS.keys():
    z_col = f"z_{metric}"
    Z_COLS[metric] = z_col

    df = df.with_columns(
        (
            (pl.col(metric) - pl.col(metric).mean()) /
            pl.col(metric).std()
        ).alias(z_col)
    )

### Compute Weighted Z-Score Contributions

In [104]:
CATEGORY_WEIGHTS = {
    "PTS_pg": 1.00,
    "Threes_pg": 1.00,
    "REB_pg": 1.00,
    "AST_pg": 1.00,
    "STL_pg": 0.95,
    "BLK_pg": 0.95,
    "FG_Impact_pg": 0.65, # This ensures efficiency matters — but does not overwhelm the model.
    "FT_Impact_pg": 0.65, # This ensures efficiency matters — but does not overwhelm the model.
}

WEIGHTED_Z_COLS = []

for metric, weight in CATEGORY_WEIGHTS.items():
    z_col = f"z_{metric}"
    w_col = f"w_{z_col}"

    WEIGHTED_Z_COLS.append(w_col)

    df = df.with_columns(
        (pl.col(z_col) * weight).alias(w_col)
    )

weighted_sum = sum(
    pl.col(metric) * weight
    for metric, weight in CATEGORY_WEIGHTS.items()
)

df = df.with_columns(
    weighted_sum.alias("Weighted_Z_Score")
)

df

Name,Pos,Games,Min,USG,PTS,Threes,REB,AST,STL,BLK,TO,Field_Goal_Percentage,Free_Throw_Percentage,Field_Goal_Attempts,Free_Throw_Attempts,FG_Impact,FT_Impact,PTS_pg,Threes_pg,REB_pg,AST_pg,STL_pg,BLK_pg,FG_Impact_pg,FT_Impact_pg,z_PTS_pg,z_Threes_pg,z_REB_pg,z_AST_pg,z_STL_pg,z_BLK_pg,z_FG_Impact_pg,z_FT_Impact_pg,w_z_PTS_pg,w_z_Threes_pg,w_z_REB_pg,w_z_AST_pg,w_z_STL_pg,w_z_BLK_pg,w_z_FG_Impact_pg,w_z_FT_Impact_pg,Weighted_Z_Score
str,str,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Nikola Jokic""","""C""",27,940.9,29.961517,794,56,327,289,37,20,96,0.605485,0.841026,474,195,57.298887,7.9128,28.8,2.074074,10.619048,7.275862,1.37037,0.740741,1.344014,0.293067,2.138864,0.452232,2.301173,2.194632,1.006135,0.27373,2.470626,1.114891,2.138864,0.452232,2.301173,2.194632,0.955828,0.260044,1.605907,0.724679,51.838642
"""Shai Gilgeous-Alexander""","""PG""",27,896.4,33.652491,878,60,126,174,39,22,52,0.554307,0.8828125,534,256,37.222797,21.085523,28.8,2.222222,4.666667,6.444444,1.444444,0.814815,1.344014,0.580906,2.138864,0.611532,-0.371788,1.740399,1.223503,0.434982,2.470626,2.257588,2.138864,0.611532,-0.371788,1.740399,1.162328,0.413233,1.605907,1.467432,45.530827
"""Tyrese Maxey""","""PG""",25,995.366667,30.319171,792,95,113,178,42,22,65,0.467354,0.884393,582,173,-10.038075,14.522638,28.8,3.307692,4.52,7.12,1.666667,0.88,-0.401523,0.580906,2.138864,1.778713,-0.43765,2.109479,1.875608,0.576883,-0.747241,2.257588,2.138864,1.778713,-0.43765,2.109479,1.781828,0.548039,-0.485707,1.467432,46.283624
"""Scottie Barnes""","""SG/SF/PF""",30,1016.566667,24.024475,572,40,240,152,41,50,79,0.497768,0.819048,448,105,5.898526,1.953046,19.066667,1.333333,8.0,5.066667,1.366667,1.642857,0.196618,0.065102,0.555985,-0.344269,1.12507,0.98767,0.995266,2.237545,0.355421,0.209888,0.555985,-0.344269,1.12507,0.98767,0.945503,2.125668,0.231024,0.136427,36.495832
"""Jalen Johnson""","""SF/PF""",27,948.966667,26.732867,642,44,284,221,40,10,93,0.524887,0.832298,442,161,17.806135,5.128005,23.777778,1.62963,10.518519,7.275862,1.481481,0.37037,0.659486,0.189926,1.322127,-0.025669,2.25603,2.194632,1.332187,-0.532528,1.208712,0.705431,1.322127,-0.025669,2.25603,2.194632,1.265578,-0.505902,0.785663,0.45853,45.513165
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Daniel Gafford""","""C""",18,386.016667,15.062784,147,0,108,22,15,29,28,0.613636,0.795918,88,49,11.355068,-0.221912,8.166667,0.0,6.0,1.222222,0.833333,1.611111,0.630837,-0.012328,-1.216624,-1.777971,0.226955,-1.112686,-0.569786,2.168438,1.155897,-0.097503,-1.216624,-1.777971,0.226955,-1.112686,-0.541297,2.060016,0.751333,-0.063377,18.113142
"""Tristan da Silva""","""SF/PF""",26,602.166667,17.369541,252,42,101,30,21,6,25,0.439252,0.814815,214,27,-9.704722,0.387926,9.692308,1.615385,3.884615,1.153846,0.807692,0.230769,-0.373259,0.01492,-0.968517,-0.040986,-0.722974,-1.150043,-0.645029,-0.836426,-0.695136,0.010672,-0.968517,-0.040986,-0.722974,-1.150043,-0.612778,-0.794605,-0.451838,0.006937,17.099772
"""Robert Williams""","""C""",18,284.183333,12.46021,107,1,108,17,12,27,11,0.680556,0.615385,72,13,14.108692,-2.405813,7.217391,0.055556,6.0,0.958333,0.666667,1.5,0.783816,-0.133656,-1.370999,-1.718234,0.226955,-1.256858,-1.058865,1.92656,1.437912,-0.579165,-1.370999,-1.718234,0.226955,-1.256858,-1.005922,1.830232,0.934643,-0.376457,16.712217


### Removing Single Worst Category

In [105]:
weighted_metrics = [
    pl.col(metric) * weight
    for metric, weight in CATEGORY_WEIGHTS.items()
]

df = df.with_columns(
    pl.concat_list(weighted_metrics)
      .list.min()
      .alias("Worst_Category_Value")
)

df = df.with_columns(
    (pl.col("Weighted_Z_Score") - pl.col("Worst_Category_Value"))
    .alias("Adjusted_Z_Score")
)

### Measuring Metric Concentration

In [106]:
df = df.with_columns(
    pl.concat_list(weighted_metrics).alias("Weighted_List")
)

df = df.with_columns(
    pl.col("Weighted_List").list.max().alias("Best_Category_Value"),
    pl.col("Weighted_List").list.sum().alias("Total_Contribution")
)

df = df.with_columns(
    (pl.col("Best_Category_Value") / pl.col("Total_Contribution"))
    .alias("Metric_Dependency")
)

# Apply soft penalty for high dependency
df = df.with_columns(
    pl.when(pl.col("Metric_Dependency") > 0.40)
      .then((pl.col("Metric_Dependency") - 0.40) * 1.5)
      .otherwise(0)
      .alias("Dependency_Penalty")
)

### Final Scoring Formula & Leaderboard

In [107]:
df = df.with_columns(
    (pl.col("Adjusted_Z_Score") - pl.col("Dependency_Penalty"))
    .alias("Final_Fantasy_Score")
)

df = (
    df.sort("Final_Fantasy_Score", descending=True)
      .with_row_index(name="Rank", offset=1)
)

df

Rank,Name,Pos,Games,Min,USG,PTS,Threes,REB,AST,STL,BLK,TO,Field_Goal_Percentage,Free_Throw_Percentage,Field_Goal_Attempts,Free_Throw_Attempts,FG_Impact,FT_Impact,PTS_pg,Threes_pg,REB_pg,AST_pg,STL_pg,BLK_pg,FG_Impact_pg,FT_Impact_pg,z_PTS_pg,z_Threes_pg,z_REB_pg,z_AST_pg,z_STL_pg,z_BLK_pg,z_FG_Impact_pg,z_FT_Impact_pg,w_z_PTS_pg,w_z_Threes_pg,w_z_REB_pg,w_z_AST_pg,w_z_STL_pg,w_z_BLK_pg,w_z_FG_Impact_pg,w_z_FT_Impact_pg,Weighted_Z_Score,Worst_Category_Value,Adjusted_Z_Score,Weighted_List,Best_Category_Value,Total_Contribution,Metric_Dependency,Dependency_Penalty,Final_Fantasy_Score
u32,str,str,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,list[f64],f64,f64,f64,f64,f64
1,"""Nikola Jokic""","""C""",27,940.9,29.961517,794,56,327,289,37,20,96,0.605485,0.841026,474,195,57.298887,7.9128,28.8,2.074074,10.619048,7.275862,1.37037,0.740741,1.344014,0.293067,2.138864,0.452232,2.301173,2.194632,1.006135,0.27373,2.470626,1.114891,2.138864,0.452232,2.301173,2.194632,0.955828,0.260044,1.605907,0.724679,51.838642,0.190493,51.648148,"[28.8, 2.074074, … 0.190493]",28.8,51.838642,0.55557,0.233355,51.414793
2,"""Luka Doncic""","""PG/SG""",21,767.783333,38.131399,716,71,181,184,33,11,88,0.458333,0.807087,480,254,-12.608722,1.686417,28.8,3.307692,8.619048,7.275862,1.571429,0.52381,-0.600415,0.080306,2.138864,1.778713,1.403058,2.194632,1.596135,-0.198507,-1.113896,0.270247,2.138864,1.778713,1.403058,2.194632,1.516328,-0.188582,-0.724032,0.17566,49.655007,-0.39027,50.045277,"[28.8, 3.307692, … 0.052199]",28.8,49.655007,0.580002,0.270003,49.775274
3,"""Giannis Antetokounmpo""","""PF/C""",17,495.3,37.816406,492,10,172,104,16,16,56,0.639456,0.634731,294,167,45.527158,-27.674679,28.8,0.588235,10.117647,6.117647,0.941176,0.941176,1.344014,-0.451696,2.138864,-1.145456,2.076016,1.561858,-0.253323,0.710058,2.470626,-1.841755,2.138864,-1.145456,2.076016,1.561858,-0.240657,0.674555,1.605907,-1.197141,47.991772,-0.293602,48.285374,"[28.8, 0.588235, … -0.293602]",28.8,47.991772,0.600103,0.300154,47.985219
4,"""Tyrese Maxey""","""PG""",25,995.366667,30.319171,792,95,113,178,42,22,65,0.467354,0.884393,582,173,-10.038075,14.522638,28.8,3.307692,4.52,7.12,1.666667,0.88,-0.401523,0.580906,2.138864,1.778713,-0.43765,2.109479,1.875608,0.576883,-0.747241,2.257588,2.138864,1.778713,-0.43765,2.109479,1.781828,0.548039,-0.485707,1.467432,46.283624,-0.26099,46.544614,"[28.8, 3.307692, … 0.377589]",28.8,46.283624,0.62225,0.333375,46.211239
5,"""Jalen Johnson""","""SF/PF""",27,948.966667,26.732867,642,44,284,221,40,10,93,0.524887,0.832298,442,161,17.806135,5.128005,23.777778,1.62963,10.518519,7.275862,1.481481,0.37037,0.659486,0.189926,1.322127,-0.025669,2.25603,2.194632,1.332187,-0.532528,1.208712,0.705431,1.322127,-0.025669,2.25603,2.194632,1.265578,-0.505902,0.785663,0.45853,45.513165,0.123452,45.389713,"[23.777778, 1.62963, … 0.123452]",23.777778,45.513165,0.522437,0.183656,45.206057
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
164,"""Josh Minott""","""SF/PF""",27,498.95,13.609102,182,32,114,32,22,11,18,0.5078125,0.769231,128,26,2.971007,-0.811627,7.217391,1.185185,4.222222,1.185185,0.814815,0.407407,0.110037,-0.03006,-1.370999,-0.503569,-0.571369,-1.132921,-0.624128,-0.451903,0.195812,-0.167897,-1.370999,-0.503569,-0.571369,-1.132921,-0.592922,-0.429307,0.127278,-0.109133,15.02308,-0.019539,15.042619,"[7.217391, 1.185185, … -0.019539]",7.217391,15.02308,0.48042,0.12063,14.921989
165,"""Craig Porter Jr.""","""PG/SG""",25,410.866667,12.592636,126,14,77,59,26,14,20,0.525253,0.533333,99,15,4.024451,-4.006708,7.217391,0.56,3.08,2.36,1.04,0.56,0.160978,-0.160268,-1.370999,-1.175816,-1.084293,-0.491078,0.036672,-0.119724,0.28972,-0.684812,-1.370999,-1.175816,-1.084293,-0.491078,0.034838,-0.113738,0.188318,-0.445128,14.737853,-0.104174,14.842027,"[7.217391, 0.56, … -0.104174]",7.217391,14.737853,0.

#### Rankings Output

In [108]:
rankings = (
    df
    .select("Name", "Pos", "Final_Fantasy_Score")
    .sort("Final_Fantasy_Score", descending=True)
)

rankings

Name,Pos,Final_Fantasy_Score
str,str,f64
"""Nikola Jokic""","""C""",51.414793
"""Luka Doncic""","""PG/SG""",49.775274
"""Giannis Antetokounmpo""","""PF/C""",47.985219
"""Tyrese Maxey""","""PG""",46.211239
"""Jalen Johnson""","""SF/PF""",45.206057
…,…,…
"""Josh Minott""","""SF/PF""",14.921989
"""Craig Porter Jr.""","""PG/SG""",14.70745
"""Will Richard""","""SG""",14.627322


### Category Contribution Analysis


In [109]:
# Categories to display (per-game metrics)
DISPLAY_COLS = [
    "PTS_pg", "Threes_pg", "REB_pg", "AST_pg",
    "STL_pg", "BLK_pg", "FG_Impact_pg", "FT_Impact_pg"
]

### Single Player Contribution Table

In [110]:
def player_category_breakdown(df: pl.DataFrame, player_name: str) -> pl.DataFrame:
    row = df.filter(pl.col("Name") == player_name)

    if row.height == 0:
        raise ValueError(f"Player '{player_name}' not found")

    # Sum of weighted z-scores before adjustments
    weighted_cols = [f"w_z_{c}" for c in DISPLAY_COLS]
    total_weighted_sum = sum(row.select(c).item() for c in weighted_cols)

    data = []

    for raw in DISPLAY_COLS:
        z_col = f"z_{raw}"
        w_col = f"w_z_{raw}"
        weight = CATEGORY_WEIGHTS[raw]

        raw_val = row.select(raw).item()
        z_val = row.select(z_col).item()
        w_val = row.select(w_col).item()

        data.append({
            "Category": raw.replace("_pg", ""),
            "Value": round(raw_val, 2),
            "Z_Score": round(z_val, 2),
            "Weight": weight,
            "Weighted_Z": round(w_val, 2),
            "Contribution_%": round((w_val / total_weighted_sum) * 100, 1),
        })

    return pl.DataFrame(data).sort("Weighted_Z", descending=True)


In [111]:
# Example: Breakdown for a specific player

pd_df = player_category_breakdown(df, "Nikola Jokic").to_pandas()

pd_df


,Category,Value,Z_Score,Weight,Weighted_Z,Contribution_%
0,REB,10.62,2.30,1.00,2.30,21.6
1,AST,7.28,2.19,1.00,2.19,20.6
2,PTS,28.80,2.14,1.00,2.14,20.1
3,FG_Impact,1.34,2.47,0.65,1.61,15.1
4,STL,1.37,1.01,0.95,0.96,9.0
5,FT_Impact,0.29,1.11,0.65,0.72,6.8
6,Threes,2.07,0.45,1.00,0.45,4.3
7,BLK,0.74,0.27,0.95,0.26,2.4


In [112]:
def zscore_color(val):
    if val > 0:
        return f"background-color: rgba(0, 160, 0, {min(abs(val)/2, 1)})"
    elif val < 0:
        return f"background-color: rgba(200, 0, 0, {min(abs(val)/2, 1)})"
    return ""


(
    pd_df
    .style
    .format({
        "Z_Score": "{:+.2f}",
        "Contribution_%": "{:+.1f}%"
    })
    .map(zscore_color, subset=["Z_Score"])
    .set_caption("Category Contributions to Fantasy Score")
)


,Category,Value,Z_Score,Weight,Weighted_Z,Contribution_%
0,REB,10.620000,+2.30,1.000000,2.300000,+21.6%
1,AST,7.280000,+2.19,1.000000,2.190000,+20.6%
2,PTS,28.800000,+2.14,1.000000,2.140000,+20.1%
3,FG_Impact,1.340000,+2.47,0.650000,1.610000,+15.1%
4,STL,1.370000,+1.01,0.950000,0.960000,+9.0%
5,FT_Impact,0.290000,+1.11,0.650000,0.720000,+6.8%
6,Threes,2.070000,+0.45,1.000000,0.450000,+4.3%
7,BLK,0.740000,+0.27,0.950000,0.260000,+2.4%


### Top-n Leaderboard with category columns

In [ ]:
# ===============================
# Fantasy Basketball Leaderboard
# ===============================

import polars as pl
import pandas as pd

# labels for categories
CATEGORY_LABELS = {
    "PTS_pg": "Points",
    "Threes_pg": "3-Pointers",
    "REB_pg": "Rebounds",
    "AST_pg": "Assists",
    "STL_pg": "Steals",
    "BLK_pg": "Blocks",
    "FG_Impact_pg": "FG% Impact",
    "FT_Impact_pg": "FT% Impact",
}

# Define z-score columns (raw)
z_cols = [
    "z_PTS_pg", "z_Threes_pg", "z_REB_pg", "z_AST_pg",
    "z_STL_pg", "z_BLK_pg", "z_FG_Impact_pg", "z_FT_Impact_pg"
]

# Weighted z-score columns (used for final fantasy score)
w_z_cols = [f"w_{col}" for col in z_cols]  # e.g., w_z_PTS_pg

# Drop old Rank if it exists
if "Rank" in df.columns:
    df = df.drop("Rank")

# Build leaderboard in Polars
leaderboard = (
    df
    .sort("Final_Fantasy_Score", descending=True)
    .with_row_index(name="Leaderboard_Rank", offset=1)  # temporary unique name
    .select(
        "Leaderboard_Rank",
        "Name",
        "Pos",
        "Final_Fantasy_Score",
        *z_cols,
        *w_z_cols
    )
    .rename({"Leaderboard_Rank": "Rank"})  # rename for display
)

# Convert to Pandas for styling
leaderboard_pd = leaderboard.to_pandas()

# Ensure numeric types are correct
for col in ["Final_Fantasy_Score"] + z_cols + w_z_cols:
    leaderboard_pd[col] = leaderboard_pd[col].astype(float)

# Compute contribution % based on weighted z-scores
weighted_sum = leaderboard_pd[w_z_cols].sum(axis=1)
for raw_col, w_col in zip(z_cols, w_z_cols):
    leaderboard_pd[f"{raw_col}_pct"] = (leaderboard_pd[w_col] / weighted_sum * 100)

# display of contribution percentages
pct_cols = [f"{col}_pct" for col in z_cols]
for col in pct_cols:
    leaderboard_pd[col + "_disp"] = leaderboard_pd[col].map(lambda x: f"{x:+.1f}%")

# Conditional formatting for z-scores
def zscore_bg(val):
    if val >= 2:
        return "background-color: rgba(0, 150, 0, 0.8)"
    elif val >= 1:
        return "background-color: rgba(0, 150, 0, 0.4)"
    elif val <= -2:
        return "background-color: rgba(200, 0, 0, 0.8)"
    elif val <= -1:
        return "background-color: rgba(200, 0, 0, 0.4)"
    return ""

# Build final display table
display_cols = ["Rank", "Name", "Pos", "Final_Fantasy_Score"] + \
               z_cols + [col + "_disp" for col in pct_cols]

# Style the table
(
    leaderboard_pd[display_cols]
    .style
    .map(zscore_bg, subset=z_cols)
    .format({"Final_Fantasy_Score": "{:.2f}"})
    .set_caption("Fantasy Basketball Leaderboard with Category Contributions")
)


,Rank,Name,Pos,Final_Fantasy_Score,z_PTS_pg,z_Threes_pg,z_REB_pg,z_AST_pg,z_STL_pg,z_BLK_pg,z_FG_Impact_pg,z_FT_Impact_pg,z_PTS_pg_pct_disp,z_Threes_pg_pct_disp,z_REB_pg_pct_disp,z_AST_pg_pct_disp,z_STL_pg_pct_disp,z_BLK_pg_pct_disp,z_FG_Impact_pg_pct_disp,z_FT_Impact_pg_pct_disp
0,1,Nikola Jokic,C,51.41,2.138864,0.452232,2.301173,2.194632,1.006135,0.273730,2.470626,1.114891,+20.1%,+4.3%,+21.6%,+20.6%,+9.0%,+2.4%,+15.1%,+6.8%
1,2,Luka Doncic,PG/SG,49.78,2.138864,1.778713,1.403058,2.194632,1.596135,-0.198507,-1.113896,0.270247,+25.8%,+21.4%,+16.9%,+26.5%,+18.3%,-2.3%,-8.7%,+2.1%
2,3,Giannis Antetokounmpo,PF/C,47.99,2.138864,-1.145456,2.076016,1.561858,-0.253323,0.710058,2.470626,-1.841755,+39.1%,-20.9%,+37.9%,+28.5%,-4.4%,+12.3%,+29.3%,-21.9%
3,4,Tyrese Maxey,PG,46.21,2.138864,1.778713,-0.437650,2.109479,1.875608,0.576883,-0.747241,2.257588,+24.0%,+20.0%,-4.9%,+23.7%,+20.0%,+6.2%,-5.5%,+16.5%
4,5,Jalen Johnson,SF/PF,45.21,1.322127,-0.025669,2.256030,2.194632,1.332187,-0.532528,1.208712,0.705431,+17.1%,-0.3%,+29.1%,+28.3%,+16.3%,-6.5%,+10.1%,+5.9%
5,6,Shai Gilgeous-Alexander,PG,44.80,2.138864,0.611532,-0.371788,1.740399,1.223503,0.434982,2.470626,2.257588,+24.4%,+7.0%,-4.2%,+19.8%,+13.3%,+4.7%,+18.3%,+16.7%
6,7,Cade Cunningham,PG/SG,44.63,1.846140,0.415593,0.388616,2.194632,1.210461,0.402731,-1.219800,0.764330,+30.4%,+6.8%,+6.4%,+36.1%,+18.9%,+6.3%,-13.0%,+8.2%
7,8,Austin Reaves,PG/SG/SF,44.10,1.970044,1.243044,0.034502,1.887823,0.059030,-0.924140,0.559266,2.257588,+32.1%,+20.2%,+0.6%,+30.7%,+0.9%,-14.3%,+5.9%,+23.9%
8,9,Donovan Mitchell,PG/SG,43.52,2.138864,1.778713,-0.429359,1.203403,1.047936,-0.668972,0.325124,1.100748,+35.8%,+29.8%,-7.2%,+20.1%,+16.7%,-10.6%,+3.5%,+12.0%
9,10,Jaylen Brown,SG/SF,43.44,2.138864,0.455295,0.347855,0.972261,0.145021,-0.417792,0.600002,-0.764381,+60.3%,+12.8%,+9.8%,+27.4%,+3.9%,-11.2%,+11.0%,-14.0%
